In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
%run /Workspace/fmcg_domain/Setup/utilities

In [0]:
bronze_schema,silver_schema,gold_schema
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("datasource","customers","Data source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("datasource")

In [0]:
silver_products_table = f"{catalog}.{silver_schema}.{data_source}"
df = spark.sql(f"select product_code,product_name,category,division,variant from {silver_products_table}")
display(df)

In [0]:
delta_table = DeltaTable.forName(spark,"fmcg.gold.dim_products")


In [0]:
df.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
            .option("delta.enableChangeDataFeed","true")\
                .saveAsTable("fmcg.gold.sb_dim_products")

df = df.dropDuplicates(subset = ["product_code"])            

In [0]:
display(df,10)

In [0]:
delta_table.alias("target").merge(
    df.alias("source"),
    condition = "target.product_code = source.product_code",
).whenMatchedUpdate(
    set = {
        "product_code": "source.product_code",
        "product": "source.product_name",
        "category": "source.category",
        "division": "source.division",
        "variant" : "source.variant"
    }
).whenNotMatchedInsert(
    values = {
        "product_code": "source.product_code",
        "product": "source.product_name",
        "category": "source.category",
        "division": "source.division",
        "variant" : "source.variant"
    }
).execute()